# Visualisation 

# 1. Imports

In [5]:
import numpy as np
import pandas as pd
import tifffile as tiff
from astropy.io import fits

from utils.naming import *

# --- Data locations ----------------------------------------------------------
GHOST = Path("/home/aosimul/noah/data/ghost_images")
RESULTS = Path("/home/aosimul/noah/src/pipeline_pre_ADI/results")

RAW_TIF_DIR = GHOST / "phase_screens_SLM/10_10"      # raw .tif frames
RAW_BENCH_DIR = GHOST / "phase_screens_SLM/10_13"    # raw .tif frames
RUNS_BENCH = GHOST / "bench"                        # batches to characterise the bench and set up noise
LONG_RUN_FOCUS = GHOST / "long_7ro_20ws_focus"      # stacked _1-4 / _5-8, DIT = 2.814 ms with focus

DIT_LONG = 2.814e-3

# 2. Function library

Taken from the temporal correlation notebook

In [6]:
def wiener_khinchin_acf(stack, subtract="median"):
    """Circular temporal ACF per pixel via the Wiener-Khinchin theorem (IFFT of the PSD).

    Normalised so that lag 0 equals 1. Returns an array shaped like `stack`."""
    x = np.asarray(stack, dtype=np.float32)
    if subtract == "median":
        x = x - np.median(x, axis=0, keepdims=True)
    elif subtract == "mean":
        x = x - np.mean(x, axis=0, keepdims=True)
    P = np.abs(np.fft.rfft(x, axis=0)) ** 2
    acf = np.fft.irfft(P, n=x.shape[0], axis=0)
    zero_lag = acf[0:1]
    return np.divide(acf, zero_lag, out=np.full_like(acf, np.nan), where=zero_lag > 0)

In [8]:
walk_stack = load_stack(LONG_RUN_FOCUS, "int", ro=7, ws=20, key="1key", count="1-4")
acf = wiener_khinchin_acf(walk_stack)